In [1]:
!git clone https://github.com/bsesic/hebrewmnist.git

fatal: destination path 'hebrewmnist' already exists and is not an empty directory.


In [2]:
import os
import re
import pandas as pd
import numpy as np
import torch
from PIL import Image
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import TensorDataset, random_split
from sklearn.preprocessing import LabelEncoder


In [3]:
'''
What need to be changed

Dataset	Size	Mode	Fix needed
Latin	28×28 (CSV) L	Fix rotation/flip
Arabic	32×32	RGB	    Resize + grayscale
Greek	28×28	L	    Nothing
Hindi	32×32	L	    Resize only
Korean	64×64	RGB	    Resize + grayscale
Hebrew	51×104	RGB	    Resize + grayscale
'''

'\nWhat need to be changed\n\nDataset\tSize\tMode\tFix needed\nLatin\t28×28 (CSV) L\tFix rotation/flip\nArabic\t32×32\tRGB\t    Resize + grayscale\nGreek\t28×28\tL\t    Nothing\nHindi\t32×32\tL\t    Resize only\nKorean\t64×64\tRGB\t    Resize + grayscale\nHebrew\t51×104\tRGB\t    Resize + grayscale\n'

# Pre Processing

In [4]:
std_transform = transforms.Compose([
    transforms.Grayscale(1),           
    transforms.Resize((28, 28)),      
    transforms.ToTensor(),             
    transforms.Lambda(lambda x: x.view(-1))  
])

In [5]:
train_df = pd.read_csv('/kaggle/input/datasets/crawford/emnist/emnist-byclass-train.csv', header=None)
test_df  = pd.read_csv('/kaggle/input/datasets/crawford/emnist/emnist-byclass-test.csv',  header=None)

y_train = torch.tensor(train_df.iloc[:, 0].values, dtype=torch.long)
x_train = torch.tensor(train_df.iloc[:, 1:].values, dtype=torch.float32) / 255.0

y_test  = torch.tensor(test_df.iloc[:, 0].values, dtype=torch.long)
x_test  = torch.tensor(test_df.iloc[:, 1:].values, dtype=torch.float32)/255.0


def fix_emnist(x_flat):
    imgs = x_flat.reshape(-1, 28, 28)     
    imgs = torch.rot90(imgs, k=1, dims=[1,2])  
    imgs = torch.flip(imgs, dims=[2])      
    return imgs.reshape(-1, 784)           

x_train = fix_emnist(x_train)
x_test  = fix_emnist(x_test)

latin_train = TensorDataset(x_train,y_train)
latin_test  = TensorDataset(x_test,y_test)

print(f"Latin  -> train: {len(latin_train)} | test: {len(latin_test)} | classes: {y_train.unique().shape[0]}")

Latin  -> train: 697932 | test: 116323 | classes: 62


**ARABIC**

In [6]:
def load_arabic(folder):
    files = sorted(f for f in os.listdir(folder) if f.endswith('.jpg'))
    imgs, raw_labels = [], []
    for f in files:
        img = Image.open(os.path.join(folder, f)).convert('L').resize((28,28))
        imgs.append(np.array(img))
        raw_labels.append(re.sub(r'\d+', '', f.replace('.jpg', '')))
    x = torch.tensor(np.stack(imgs), dtype=torch.float32) / 255.0
    return x.reshape(-1, 784), raw_labels

x_tr, lbl_tr = load_arabic('/kaggle/input/datasets/rashwan/arabic-chars-mnist/train')
x_te, lbl_te = load_arabic('/kaggle/input/datasets/rashwan/arabic-chars-mnist/test')
le_ar        = LabelEncoder().fit(lbl_tr)
arabic_train = TensorDataset(x_tr, torch.tensor(le_ar.transform(lbl_tr), dtype=torch.long))
arabic_test  = TensorDataset(x_te, torch.tensor(le_ar.transform(lbl_te), dtype=torch.long))
print(f"Arabic  -> train: {len(arabic_train)} | test: {len(arabic_test)} | classes: {len(le_ar.classes_)} ")


Arabic  -> train: 13440 | test: 3360 | classes: 28 


**Greek...**

In [7]:
greek_data          = ImageFolder('/kaggle/input/datasets/sayangupta001/mnist-greek-letters/Greek_Letters', std_transform)
tr_n, te_n          = int(0.8*len(greek_data)), len(greek_data)-int(0.8*len(greek_data))
greek_train, greek_test = random_split(greek_data, [tr_n, te_n])
print(f"Greek   -> train: {len(greek_train)} | test: {len(greek_test)} | classes: {len(greek_data.classes)} ")


Greek   -> train: 480 | test: 120 | classes: 24 


**HINDI...**

In [8]:
hindi_train = ImageFolder('/kaggle/input/datasets/berlinsweird/devanagari/Hindi/Train', std_transform)
hindi_test  = ImageFolder('/kaggle/input/datasets/berlinsweird/devanagari/Hindi/Test',  std_transform)
print(f"Hindi   -> train: {len(hindi_train)} | test: {len(hindi_test)} | classes: {len(hindi_train.classes)} ")

Hindi   -> train: 78200 | test: 13800 | classes: 46 


**Korean...**

In [9]:
korean_data          = ImageFolder('/kaggle/input/datasets/jkim289/handwritten-korean-characters/Hangul Database/Hangul Database', std_transform)
tr_n, te_n           = int(0.8*len(korean_data)), len(korean_data)-int(0.8*len(korean_data))
korean_train, korean_test = random_split(korean_data, [tr_n, te_n])
print(f"Korean  -> train: {len(korean_train)} | test: {len(korean_test)} | classes: {len(korean_data.classes)} ")


Korean  -> train: 5120 | test: 1280 | classes: 64 


**HEBREW**

In [10]:
hebrew_data          = ImageFolder('/kaggle/working/hebrewmnist/hebrew_letters', std_transform)
tr_n, te_n           = int(0.8*len(hebrew_data)), len(hebrew_data)-int(0.8*len(hebrew_data))
hebrew_train, hebrew_test = random_split(hebrew_data, [tr_n, te_n])
print(f"Hebrew  -> train: {len(hebrew_train)} | test: {len(hebrew_test)} | classes: {len(hebrew_data.classes)} ")

Hebrew  -> train: 245 | test: 62 | classes: 28 


# DataLoaders

In [11]:
from torch.utils.data import DataLoader
from itertools import cycle

BATCH_SIZE = 32

train_loaders = [
    DataLoader(latin_train,  batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(arabic_train, batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(greek_train,  batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(hindi_train,  batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(korean_train, batch_size=BATCH_SIZE, shuffle=True),
    DataLoader(hebrew_train, batch_size=BATCH_SIZE, shuffle=True),
]

test_loaders = [
    DataLoader(latin_test,  batch_size=256, shuffle=False),
    DataLoader(arabic_test, batch_size=256, shuffle=False),
    DataLoader(greek_test,  batch_size=256, shuffle=False),
    DataLoader(hindi_test,  batch_size=256, shuffle=False),
    DataLoader(korean_test, batch_size=256, shuffle=False),
    DataLoader(hebrew_test, batch_size=256, shuffle=False),
]

#as in our dataset the latin has more batches 
#so others like herbew will finish earlier to avoid that I put inf_iters it will loop
inf_iters = [cycle(loader) for loader in train_loaders]

SCRIPT_NAMES   = ["Latin","Arabic","Greek","Hindi","Korean","Hebrew"]
SCRIPT_CLASSES = [62, 28, 24, 46, 64, 28]

for i, (name, loader) in enumerate(zip(SCRIPT_NAMES, train_loaders)):
    print(f"  m={i} {name}: {len(loader)} batches per epoch")

  m=0 Latin: 21811 batches per epoch
  m=1 Arabic: 420 batches per epoch
  m=2 Greek: 15 batches per epoch
  m=3 Hindi: 2444 batches per epoch
  m=4 Korean: 160 batches per epoch
  m=5 Hebrew: 8 batches per epoch


# Defining Network

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
INPUT_SIZE     = 784
HIDDEN_SIZE    = 128
NUM_G_SETS     = 4
SCRIPT_NAMES   = ["Latin","Arabic","Greek","Hindi","Korean","Hebrew"]
SCRIPT_CLASSES = [62, 28, 24, 46, 64, 28]

B_CONFIGS = torch.tensor([
    [1,1,0,0],   #Latin
    [1,0,1,0],   #Arabic
    [1,0,0,1],   #Greek
    [0,1,1,0],   #Hindi
    [0,1,0,1],   #Korean
    [0,0,1,1],   #Hebrew
], dtype=torch.float32).to(DEVICE)


#G values (compound synapses)
G = nn.Parameter(
    torch.randn(NUM_G_SETS, HIDDEN_SIZE, INPUT_SIZE, device=DEVICE) * 0.01
)

#output heads
head_latin  = nn.Linear(HIDDEN_SIZE, 62).to(DEVICE)
head_arabic = nn.Linear(HIDDEN_SIZE, 28).to(DEVICE)
head_greek  = nn.Linear(HIDDEN_SIZE, 24).to(DEVICE)
head_hindi  = nn.Linear(HIDDEN_SIZE, 46).to(DEVICE)
head_korean = nn.Linear(HIDDEN_SIZE, 64).to(DEVICE)
head_hebrew = nn.Linear(HIDDEN_SIZE, 28).to(DEVICE)

heads = [head_latin, head_arabic, head_greek,
         head_hindi, head_korean, head_hebrew]

def forward(x, m):
    b = B_CONFIGS[m]                         
    W = torch.einsum('n,nhi->hi', b, G)      
    h = F.relu(x @ W.T)                      
    return heads[m](h)                  

#all trainable parameters 
all_params = [G] + [p for head in heads for p in head.parameters()]


# Training Setup

In [13]:
LEARNING_RATE   = 0.001
NUM_EPOCHS      = 100
STEPS_PER_EPOCH = 500  

optimizer = torch.optim.Adam(all_params, lr=LEARNING_RATE)

In [14]:
print("Training...")
print(f"Device: {DEVICE}")

for epoch in range(NUM_EPOCHS):

    total_loss = 0.0

    for step in range(STEPS_PER_EPOCH):

        optimizer.zero_grad()

        step_loss = torch.tensor(0.0, device=DEVICE)

        for m in range(6):
            x, y = next(inf_iters[m])
            x = x.float().to(DEVICE)
            y = y.to(DEVICE)

            out  = forward(x, m)
            loss = F.cross_entropy(out, y)

            step_loss = step_loss + loss

        # one backprop on total loss
        step_loss.backward()
        optimizer.step()

        total_loss += step_loss.item()

    avg_loss = total_loss / STEPS_PER_EPOCH

    if (epoch + 1) % 5 == 0:
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | avg loss: {avg_loss:.4f}")

        for m, name in enumerate(SCRIPT_NAMES):
            correct = 0
            total   = 0
            with torch.no_grad():
                for x, y in test_loaders[m]:
                    x = x.float().to(DEVICE)
                    y = y.to(DEVICE)
                    out   = forward(x, m)
                    preds = out.argmax(dim=1)
                    correct += (preds == y).sum().item()
                    total   += y.size(0)
            acc = correct / total * 100
            print(f"  {name:8s}: {acc:.2f}%")

print("\nTraining complete")

Training...
Device: cuda

Epoch 5/100 | avg loss: 4.0277
  Latin   : 74.91%
  Arabic  : 57.35%
  Greek   : 80.83%
  Hindi   : 81.62%
  Korean  : 63.05%
  Hebrew  : 87.10%

Epoch 10/100 | avg loss: 2.8884
  Latin   : 77.66%
  Arabic  : 65.30%
  Greek   : 81.67%
  Hindi   : 85.28%
  Korean  : 65.47%
  Hebrew  : 85.48%

Epoch 15/100 | avg loss: 2.3889
  Latin   : 79.10%
  Arabic  : 68.10%
  Greek   : 82.50%
  Hindi   : 87.60%
  Korean  : 67.89%
  Hebrew  : 82.26%

Epoch 20/100 | avg loss: 2.0644
  Latin   : 79.58%
  Arabic  : 68.84%
  Greek   : 82.50%
  Hindi   : 88.43%
  Korean  : 66.02%
  Hebrew  : 83.87%

Epoch 25/100 | avg loss: 1.7957
  Latin   : 79.81%
  Arabic  : 70.06%
  Greek   : 83.33%
  Hindi   : 89.24%
  Korean  : 71.64%
  Hebrew  : 83.87%

Epoch 30/100 | avg loss: 1.6177
  Latin   : 79.67%
  Arabic  : 70.68%
  Greek   : 83.33%
  Hindi   : 89.70%
  Korean  : 68.83%
  Hebrew  : 83.87%

Epoch 35/100 | avg loss: 1.5083
  Latin   : 80.28%
  Arabic  : 68.90%
  Greek   : 81.67%
  Hi